# 🩸 Twitter Sentiment Analysis
### Black & Red Edition

**Author:** Ankit Jinkwan | **Portfolio:** [ankitjhinkwan.github.io/portfolio](https://ankitjhinkwan.github.io/portfolio/)

---

### 🎯 Goal
Analyse Twitter sentiment across topics, languages, and time using NLP-style scoring.

### 📂 Dataset
- 3,000 tweets | 12 topics | 5 languages | full 2023

### 🔧 Steps
1. Import Libraries
2. Load & Explore
3. Sentiment Distribution
4. Topic Analysis
5. Engagement Analysis
6. Temporal Patterns
7. Language & Device
8. Key Insights

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Blood & Fire dark theme
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0a0000',
    'axes.facecolor':   '#0f0000',
    'axes.edgecolor':   '#8b0000',
    'axes.labelcolor':  '#cc4444',
    'xtick.color':      '#cc4444',
    'ytick.color':      '#cc4444',
    'text.color':       '#cc4444',
    'grid.color':       '#1a0000',
    'figure.figsize':   (14, 5)
})
RED   = '#ff0000'
DARK  = '#8b0000'
GREEN = '#00ff41'
GREY  = '#888888'
print('🩸 Blood mode activated!')

## Step 2 — Load & Explore

In [ ]:
df = pd.read_csv('../data/twitter_data.csv')
print(f'Total Tweets: {len(df):,}')
print(f'Positive:     {(df["Sentiment"]=="Positive").sum():,} ({(df["Sentiment"]=="Positive").mean():.1%})')
print(f'Negative:     {(df["Sentiment"]=="Negative").sum():,} ({(df["Sentiment"]=="Negative").mean():.1%})')
print(f'Neutral:      {(df["Sentiment"]=="Neutral").sum():,} ({(df["Sentiment"]=="Neutral").mean():.1%})')
print(f'Topics:       {df["Topic"].nunique()}')
print(f'Languages:    {df["Language"].nunique()}')
df.head()

## Step 3 — Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor('#0a0000')

# Pie
sc = df['Sentiment'].value_counts()
axes[0].pie(sc.values, labels=sc.index, colors=[GREEN,RED,GREY],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='#0a0000', linewidth=3),
            textprops=dict(color='white'))
axes[0].set_title('Sentiment Distribution', color=RED, fontweight='bold')
axes[0].set_facecolor('#0a0000')

# Score histogram
for sent, color in [('Positive',GREEN),('Negative',RED),('Neutral',GREY)]:
    axes[1].hist(df[df['Sentiment']==sent]['SentimentScore'], bins=25, alpha=0.7, label=sent, color=color)
axes[1].axvline(0, color='white', lw=1.5, linestyle='--')
axes[1].set_title('Sentiment Score Distribution', color=RED, fontweight='bold')
axes[1].set_xlabel('Score (-1 to +1)'); axes[1].legend()
axes[1].set_facecolor('#0f0000')

# Avg score by topic
topic_score = df.groupby('Topic')['SentimentScore'].mean().sort_values()
colors = [GREEN if v > 0 else RED for v in topic_score.values]
axes[2].barh(topic_score.index, topic_score.values, color=colors, edgecolor='#0a0000')
axes[2].axvline(0, color='white', lw=1.5, linestyle='--')
axes[2].set_title('Avg Score by Topic', color=RED, fontweight='bold')
axes[2].set_facecolor('#0f0000')

plt.tight_layout(); plt.savefig('../sentiment_overview.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 4 — Topic Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0a0000')

# Stacked bar
tp = df.groupby(['Topic','Sentiment']).size().unstack(fill_value=0)
tp.plot(kind='bar', stacked=True, ax=axes[0], color=[GREEN,RED,GREY], edgecolor='#0a0000', linewidth=0.5)
axes[0].set_title('Tweet Count by Topic & Sentiment', color=RED, fontweight='bold')
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=30)
axes[0].set_facecolor('#0f0000')
axes[0].legend(loc='upper right')

# Top topics
topic_counts = df['Topic'].value_counts()
cols = plt.cm.Reds(np.linspace(0.4, 1, len(topic_counts)))
axes[1].barh(topic_counts.index[::-1], topic_counts.values[::-1], color=cols[::-1], edgecolor='#0a0000')
axes[1].set_title('Total Tweets by Topic', color=RED, fontweight='bold')
axes[1].set_facecolor('#0f0000')
for i,v in enumerate(topic_counts.values[::-1]):
    axes[1].text(v+5, i, str(v), va='center', color=RED, fontweight='bold', fontsize=9)

plt.tight_layout(); plt.savefig('../topic_analysis.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 5 — Engagement Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0000')

# Engagement by sentiment
eng = df.groupby('Sentiment').agg(Likes=('Likes','mean'),Retweets=('Retweets','mean'),Replies=('Replies','mean')).round(1)
x = np.arange(len(eng))
w = 0.25
for i,(col,color) in enumerate(zip(['Likes','Retweets','Replies'],[RED,'#ff6600',DARK])):
    axes[0].bar(x + i*w, eng[col], w, label=col, color=color, edgecolor='#0a0000')
axes[0].set_xticks(x + w); axes[0].set_xticklabels(eng.index)
axes[0].set_title('Avg Engagement by Sentiment', color=RED, fontweight='bold')
axes[0].set_facecolor('#0f0000'); axes[0].legend()

# Likes distribution
for sent, color in [('Positive',GREEN),('Negative',RED),('Neutral',GREY)]:
    data = df[df['Sentiment']==sent]['Likes'].clip(0,300)
    axes[1].hist(data, bins=30, alpha=0.6, label=sent, color=color)
axes[1].set_title('Likes Distribution by Sentiment', color=RED, fontweight='bold')
axes[1].set_xlabel('Likes'); axes[1].legend()
axes[1].set_facecolor('#0f0000')

plt.tight_layout(); plt.savefig('../engagement.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 6 — Temporal Patterns

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M').astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0000')

# Monthly trend
monthly = df.groupby(['Month','Sentiment']).size().unstack(fill_value=0).reset_index()
for sent, color in [('Positive',GREEN),('Negative',RED),('Neutral',GREY)]:
    if sent in monthly.columns:
        axes[0].plot(range(len(monthly)), monthly[sent], color=color, lw=2, marker='o', ms=4, label=sent)
        axes[0].fill_between(range(len(monthly)), monthly[sent], alpha=0.15, color=color)
axes[0].set_xticks(range(len(monthly)))
axes[0].set_xticklabels(monthly['Month'], rotation=45, ha='right', fontsize=8)
axes[0].set_title('Monthly Sentiment Trend', color=RED, fontweight='bold')
axes[0].set_facecolor('#0f0000'); axes[0].legend()

# Hourly
hourly = df.groupby('Hour')['SentimentScore'].mean()
axes[1].plot(hourly.index, hourly.values, color=RED, lw=2.5, marker='o', ms=5)
axes[1].fill_between(hourly.index, hourly.values, alpha=0.2, color=RED)
axes[1].axhline(0, color='white', lw=1, linestyle='--')
axes[1].set_title('Avg Sentiment Score by Hour', color=RED, fontweight='bold')
axes[1].set_xlabel('Hour of Day'); axes[1].set_facecolor('#0f0000')

plt.tight_layout(); plt.savefig('../temporal.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 7 — Language & Device

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0000')

lang = df['Language'].value_counts()
cols = plt.cm.Reds(np.linspace(0.4, 1, len(lang)))
axes[0].bar(lang.index, lang.values, color=cols, edgecolor='#0a0000')
axes[0].set_title('Tweets by Language', color=RED, fontweight='bold')
axes[0].set_facecolor('#0f0000')
for i,v in enumerate(lang.values): axes[0].text(i, v+5, str(v), ha='center', color=RED, fontweight='bold')

dev = df['Device'].value_counts()
axes[1].pie(dev.values, labels=dev.index, colors=[RED,DARK,'#cc4400','#660000'],
            autopct='%1.1f%%', startangle=90,
            wedgeprops=dict(edgecolor='#0a0000', linewidth=3),
            textprops=dict(color='white'))
axes[1].set_title('Device Distribution', color=RED, fontweight='bold')
axes[1].set_facecolor('#0a0000')

plt.tight_layout(); plt.savefig('../language_device.png', dpi=150, bbox_inches='tight'); plt.show()

## Step 8 — Key Insights

In [ ]:
print('='*55)
print('     TWITTER SENTIMENT ANALYSIS — KEY INSIGHTS')
print('='*55)
print(f'  Total Tweets:       {len(df):,}')
print(f'  Positive Rate:      {(df["Sentiment"]=="Positive").mean():.1%}')
print(f'  Negative Rate:      {(df["Sentiment"]=="Negative").mean():.1%}')
print(f'  Avg Sentiment:      {df["SentimentScore"].mean():.3f}')
print(f'  Most Positive Topic:{df.groupby("Topic")["SentimentScore"].mean().idxmax()}')
print(f'  Most Negative Topic:{df.groupby("Topic")["SentimentScore"].mean().idxmin()}')
print(f'  Top Language:       {df["Language"].value_counts().index[0]}')
print(f'  Top Device:         {df["Device"].value_counts().index[0]}')
print(f'  Peak Hour:          {df.groupby("Hour").size().idxmax()}:00')
print(f'  Avg Likes/Tweet:    {df["Likes"].mean():.0f}')
print('='*55)
print('\n🩸 Run: streamlit run app/app.py')

## ✅ Summary

| Metric | Value |
|--------|-------|
| Total Tweets | 3,000 |
| Positive Rate | ~42% |
| Negative Rate | ~33% |
| Top Topic | Technology |
| Top Language | English |

### 🔑 Key Findings
- **Technology & Entertainment** generate most positive sentiment
- **Politics & Economy** have highest negative sentiment
- **Mobile** is the dominant device (55%+)
- **Evening hours** (6–10pm) see peak tweet activity
- **Negative tweets** get more engagement than positive ones

---
*🩸 Built by Ankit Jinkwan | [Portfolio](https://ankitjhinkwan.github.io/portfolio/)*